# Lab | Error Handling

Objective: Practice how to identify, handle and recover from potential errors in Python code using try-except blocks.

## Challenge 

Paste here your lab *functions* solutions. Apply error handling techniques to each function using try-except blocks. 

The try-except block in Python is designed to handle exceptions and provide a fallback mechanism when code encounters errors. By enclosing the code that could potentially throw errors in a try block, followed by specific or general exception handling in the except block, we can gracefully recover from errors and continue program execution.

However, there may be cases where an input may not produce an immediate error, but still needs to be addressed. In such situations, it can be useful to explicitly raise an error using the "raise" keyword, either to draw attention to the issue or handle it elsewhere in the program.

Modify the code to handle possible errors in Python, it is recommended to use `try-except-else-finally` blocks, incorporate the `raise` keyword where necessary, and print meaningful error messages to alert users of any issues that may occur during program execution.



In [ ]:
# ================= Excepciones personalizadas =================
class InvalidQuantityError(ValueError):
    pass

class InvalidPriceError(ValueError):
    pass

class InvalidProductError(ValueError):
    pass

class OutOfStockError(RuntimeError):
    pass


# ================= Utilidades de entrada segura =================
def pedir_entero_no_negativo(prompt: str) -> int:
    """Devuelve un entero >= 0. Aplica try/except/else/finally con mensajes claros."""
    while True:
        try:
            texto = input(prompt)
            valor = int(texto)
            if valor < 0:
                raise InvalidQuantityError("El valor no puede ser negativo.")
        except ValueError:
            print("Error: introduce un número entero válido (ej. 0, 1, 2...).")
        except InvalidQuantityError as e:
            print(f"Error: {e}")
        else:
            # Solo si no hubo excepción
            return valor
        finally:
            # Se ejecuta siempre (útil para trazas/separadores/limpiezas)
            print("— intento procesado —")


def pedir_entero_positivo(prompt: str) -> int:
    while True:
        try:
            texto = input(prompt)
            valor = int(texto)
            if valor <= 0:
                raise InvalidQuantityError("Debe ser un entero positivo.")
        except (ValueError, InvalidQuantityError) as e:
            print(f"Error: {e}")
        else:
            return valor
        finally:
            print("— intento procesado —")


def pedir_float_positivo(prompt: str) -> float:
    while True:
        try:
            texto = input(prompt).replace(",", ".")
            valor = float(texto)
            if valor <= 0:
                raise InvalidPriceError("El precio debe ser mayor que 0.")
        except ValueError:
            print("Error: introduce un número (puedes usar coma o punto).")
        except InvalidPriceError as e:
            print(f"Error: {e}")
        else:
            return valor
        finally:
            print("— intento procesado —")


# ================= Lógica del dominio =================
def initialize_inventory(products):
    """Inicializa el inventario pidiendo cantidades con manejo de errores."""
    inventory = {}
    print("=== Inicialización del inventario ===")
    for product in products:
        qty = pedir_entero_no_negativo(f"Cantidad disponible de {product}: ")
        inventory[product] = qty
    return inventory


def get_customer_orders(inventory, products):
    """
    Solicita N pedidos. Valida que:
    - N sea entero positivo.
    - El producto exista.
    - Haya stock disponible.
    Demostración de raise InvalidProductError y OutOfStockError.
    """
    print("\n=== Pedidos del cliente ===")
    num_orders = pedir_entero_positivo("Número de unidades a pedir: ")

    pedidos = []
    productos_validos = [p.lower() for p in products]

    for i in range(num_orders):
        while True:
            try:
                prod = input(f"Producto #{i+1}: ").strip().lower()
                if prod not in productos_validos:
                    raise InvalidProductError(
                        f"Producto inválido. Elige entre: {', '.join(products)}."
                    )
                if inventory.get(prod, 0) <= 0:
                    raise OutOfStockError(
                        f"No hay stock de '{prod}'. Stock actual: {inventory.get(prod, 0)}."
                    )
            except (InvalidProductError, OutOfStockError) as e:
                print(f"Error: {e}")
            else:
                pedidos.append(prod)
                break
            finally:
                # útil para trazar cada intento
                print("— intento de selección procesado —")
    return pedidos


def total_price_customer_orders(customer_orders):
    """
    Pide el precio unitario de cada producto distinto y calcula el total.
    Usa else para sumar solo cuando no hay error.
    """
    print("\n=== Precios ===")
    from collections import Counter
    cuenta = Counter(customer_orders)
    total = 0.0

    for producto, unidades in cuenta.items():
        while True:
            try:
                precio = pedir_float_positivo(f"Precio unitario de {producto}: ")
            except Exception as e:
                # En principio pedir_float_positivo ya maneja todo,
                # pero dejamos el catch por si se modifica a futuro.
                print(f"Error inesperado pidiendo precio: {e}")
            else:
                total += precio * unidades
                break
            finally:
                print("— intento de precio procesado —")
    return total


def update_inventory(customer_orders, inventory):
    """Descuenta 1 unidad por cada ocurrencia del producto en customer_orders."""
    from collections import Counter
    cuenta = Counter(customer_orders)

    for producto, unidades in cuenta.items():
        stock = inventory.get(producto, 0)
        # Si por lógica del programa llegamos aquí sin stock suficiente,
        # sería un error de consistencia → raise
        if unidades > stock:
            raise OutOfStockError(
                f"Inconsistencia: intentas descontar {unidades} de '{producto}' pero solo hay {stock}."
            )
        inventory[producto] = stock - unidades
    return inventory


def calculate_order_statistics(customer_orders, initial_total_units):
    unidades_pedidas = len(customer_orders)
    porcentaje = (unidades_pedidas / initial_total_units * 100) if initial_total_units > 0 else 0.0
    return unidades_pedidas, porcentaje


def print_order_statistics(stats):
    print(f"""
=== Estadísticas del pedido ===
Unidades pedidas: {stats[0]}
% sobre el inventario inicial: {stats[1]:.2f}%
""")


def print_updated_inventory(inventory):
    print("=== Inventario actualizado ===")
    for prod, qty in inventory.items():
        print(f"{prod}: {qty}")


# ================= Programa principal con manejo global =================
if __name__ == "__main__":
    try:
        products = ["t-shirt", "mug", "hat", "book", "keychain"]
        inventory = initialize_inventory(products)
        initial_total_units = sum(inventory.values())

        orders = get_customer_orders(inventory, products)

        stats = calculate_order_statistics(orders, initial_total_units)
        print_order_statistics(stats)

        inventory = update_inventory(orders, inventory)
        print_updated_inventory(inventory)

        total = total_price_customer_orders(orders)
        print(f"\nTotal a pagar: {total:.2f}")

    except KeyboardInterrupt:
        print("\nOperación cancelada por el usuario (Ctrl+C).")
    except Exception as e:
        # Captura final: evita que el programa “reviente” sin mensaje
        print(f"\nSe produjo un error inesperado: {type(e).__name__}: {e}")
        print("Por favor, revisa los datos de entrada e inténtalo de nuevo.")
    finally:
        # Siempre se ejecuta, haya o no errores
        print("\n— Ejecución finalizada —")


=== Inicialización del inventario ===

=== Pedidos del cliente ===
Error: Producto inválido. Elige entre: t-shirt, mug, hat, book, keychain.
Error: Producto inválido. Elige entre: t-shirt, mug, hat, book, keychain.
Error: Producto inválido. Elige entre: t-shirt, mug, hat, book, keychain.
Error: Producto inválido. Elige entre: t-shirt, mug, hat, book, keychain.
Error: Producto inválido. Elige entre: t-shirt, mug, hat, book, keychain.
Error: Producto inválido. Elige entre: t-shirt, mug, hat, book, keychain.
Error: Producto inválido. Elige entre: t-shirt, mug, hat, book, keychain.
Error: Producto inválido. Elige entre: t-shirt, mug, hat, book, keychain.
Error: Producto inválido. Elige entre: t-shirt, mug, hat, book, keychain.
Error: Producto inválido. Elige entre: t-shirt, mug, hat, book, keychain.
Error: Producto inválido. Elige entre: t-shirt, mug, hat, book, keychain.
Error: Producto inválido. Elige entre: t-shirt, mug, hat, book, keychain.
Error: Producto inválido. Elige entre: t-shir